***

* [总目录](../0_Introduction/0_introduction.ipynb)
* [术语表](../0_Introduction/1_glossary.ipynb)
* [5. 成像](5_0_introduction.ipynb)

***


导入标准模块:

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

NOTEBOOK_DIR = Path("5_Imaging") if Path("5_Imaging").exists() else Path(".")
NOTEBOOK_DIR = NOTEBOOK_DIR.resolve()
TOGGLE_PATH = NOTEBOOK_DIR.parent / "style" / "code_toggle.html"

In [ ]:
try:
    from IPython.display import HTML, Javascript
except ImportError:
    def HTML(*args, **kwargs):
        return None

    def Javascript(*args, **kwargs):
        return None

if TOGGLE_PATH.exists():
    HTML(TOGGLE_PATH.read_text(encoding="utf-8"))

***

## 5.A 综合成像中的匹配滤波器<a id='imaging:sec:matched'></a>


## 5.A 综合成像中的匹配滤波器<a id='imaging:sec:matched'></a>

匹配滤波器是一类用来在噪声背景中最大化目标信号信噪比的优化滤波器。它的基本思想，是用目标信号的模板与含噪输入做卷积，从而突出关心的结构。在综合成像里，这个思想常出现在两个地方。第一，当目标具有大致已知的形状或角尺度时，可以用匹配滤波增强这类结构的可检出性。第二，当模型源被用于自校准或快速搜索时，模板越接近真实天空，得到的响应通常越尖锐、越稳定。

从 Fourier 角度看，匹配滤波器并不是一个额外附加的“技巧”，而是把先验模板直接嵌入相关运算。若模板已知，卷积后的响应在目标位置会达到最大值；若模板与真实源不匹配，峰值仍常常保留，但增益会下降，旁瓣和背景响应也可能抬高。这个附录用几个简单例子说明这种行为。


### 5.A.1 高斯模板

先构造一个二维高斯源，再向图像中加入高斯噪声。随后将与该源形状相似的高斯模板作为滤波器，对含噪图像做卷积。这样的设置对应“模板大致已知，但测量噪声很强”的常见情形。


In [ ]:
def generalGauss2d(x0, y0, sigmax, sigmay, amp=1.0, theta=0.0):
    """Return a normalized general 2-D Gaussian function.

    Parameters
    ----------
    x0, y0 : float
        Centre position.
    sigmax, sigmay : float
        Standard deviation along each principal axis.
    amp : float
        Peak amplitude.
    theta : float
        Rotation angle in degrees.
    """
    norm = amp
    rtheta = np.deg2rad(theta)

    a = (np.cos(rtheta) ** 2.0) / (2.0 * sigmax ** 2.0) + (np.sin(rtheta) ** 2.0) / (2.0 * sigmay ** 2.0)
    b = -(np.sin(2.0 * rtheta)) / (4.0 * sigmax ** 2.0) + (np.sin(2.0 * rtheta)) / (4.0 * sigmay ** 2.0)
    c = (np.sin(rtheta) ** 2.0) / (2.0 * sigmax ** 2.0) + (np.cos(rtheta) ** 2.0) / (2.0 * sigmay ** 2.0)
    return lambda x, y: norm * np.exp(-1.0 * (a * (x - x0) ** 2.0 - 2.0 * b * (x - x0) * (y - y0) + c * (y - y0) ** 2.0))



def apply_matched_filter(template, image):
    template = template / np.sqrt(np.sum(template ** 2))
    return np.fft.ifft2(np.fft.fft2(np.fft.fftshift(template)) * np.fft.fft2(image)).real



def estimate_peak_snr(image, mask):
    background = image[~mask]
    return float(np.max(image[mask]) / np.std(background))

In [ ]:
rng = np.random.default_rng(4)
imgSize = 512
sizeScale = 33.0
xpos, ypos = np.mgrid[0:imgSize, 0:imgSize].astype(float)

gFunc = generalGauss2d(300, 100, 7.0, 10.0, amp=1.0, theta=sizeScale)
trueSignal = gFunc(xpos, ypos)
single_source_mask = trueSignal > 0.25 * np.max(trueSignal)

fig = plt.figure(figsize=(9, 9))
plt.imshow(trueSignal, origin='lower')
plt.colorbar(shrink=0.75)
plt.title('Original source')

*图：二维高斯源。*


加入零均值高斯噪声后，目标峰值信噪比明显下降。匹配滤波器的作用，就是把与模板一致的结构从背景中重新抬出来。


In [ ]:
SNR = 0.3
noise_std = np.max(trueSignal) / SNR
noisySignal = trueSignal + noise_std * rng.standard_normal(trueSignal.shape)
print(f"Peak SNR before filtering = {estimate_peak_snr(noisySignal, single_source_mask):.3f}")

fig = plt.figure(figsize=(9, 9))
plt.imshow(noisySignal / np.max(np.abs(noisySignal)), origin='lower')
plt.colorbar(shrink=0.75)
plt.title('Original source with Gaussian noise added')

*图：信噪比为 0.3 的二维高斯源。*


已知模板的形状之后，可以直接把模板作为卷积核。模板放在滤波器图像中央，卷积响应的峰值就会对应目标位置。


In [ ]:
gFunc1 = generalGauss2d(imgSize / 2.0, imgSize / 2.0, 7.0, 10.0, amp=1.0, theta=33.0)
matchedFilter = gFunc1(xpos, ypos)
fig = plt.figure(figsize=(9, 9))
plt.imshow(matchedFilter, origin='lower')
plt.colorbar(shrink=0.75)
plt.title('Matched filter template')

*图：用于该源的匹配滤波器。*


将模板与含噪图像做卷积后，目标附近会出现明显的峰值增强。


In [ ]:
filteredSignal = apply_matched_filter(matchedFilter, noisySignal)
print(f"Peak SNR after matched filtering = {estimate_peak_snr(filteredSignal, single_source_mask):.3f}")

fig = plt.figure(figsize=(9, 9))
plt.imshow(filteredSignal / np.max(np.abs(filteredSignal)), origin='lower')
plt.colorbar(shrink=0.75)
plt.title('Matched filter applied')

*图：施加匹配滤波器后的结果图像。*


即使噪声较强，合适的模板仍能让目标结构在匹配滤波响应中显著凸显出来。


### 5.A.2 模板失配

实际问题中，模板往往并不完全正确。尺度、朝向、轴比和位置都可能与真实源存在偏差。即便如此，近似匹配的模板仍然通常能显著提高目标的峰值响应，只是提升幅度低于完全匹配的情形。


模板不完全匹配时，峰值通常仍然存在，但增益较低，且响应形状更宽。


In [ ]:
gFunc1 = generalGauss2d(imgSize / 2.0, imgSize / 2.0, 2.0, 3.0, amp=1.0, theta=0.0)
matchedFilter = gFunc1(xpos, ypos)
fig = plt.figure(figsize=(8, 8))
plt.imshow(matchedFilter, origin='lower')
plt.title('Mismatched filter template')

In [ ]:
filteredSignal = apply_matched_filter(matchedFilter, noisySignal)
fig = plt.figure(figsize=(8, 8))
plt.imshow(filteredSignal, origin='lower')
plt.title('Mismatched filter applied')

### 5.A.3 多源情形

当图像中含有多个结构不同的源时，匹配滤波器对某一类源的响应会明显更强，而对其他源的响应则更弱。这样做的效果，和在可见度域里只强调某一类空间尺度很相似：滤波器实际上把相关先验放大了。

在自校准和快速源检索中，这种“按模板增强某一类结构”的性质很有用。若模板来自较简单的紧致源模型，滤波后的图像通常更容易用于增益求解；若模板针对扩展源，则可以突出更大尺度的辐射结构。


In [ ]:
gFunc = generalGauss2d(300, 100, 7.0, 10.0, amp=1.0, theta=33.0)
trueSignal = gFunc(xpos, ypos)

gFunc = generalGauss2d(400, 400, 20.0, 15.0, amp=0.3, theta=0.0)
trueSignal += gFunc(xpos, ypos)

gFunc = generalGauss2d(200, 300, 2.0, 3.0, amp=2.0, theta=45.0)
trueSignal += gFunc(xpos, ypos)

fig = plt.figure(figsize=(8, 8))
plt.imshow(trueSignal, origin='lower')
plt.title('Multi-source model image')

In [ ]:
SNR = 0.3
noise_std = np.max(trueSignal) / SNR
noisySignal = trueSignal + noise_std * rng.standard_normal(trueSignal.shape)
fig = plt.figure(figsize=(8, 8))
plt.imshow(noisySignal, origin='lower')
plt.title('Multi-source image with Gaussian noise')

In [ ]:
gFunc1 = generalGauss2d(imgSize / 2.0, imgSize / 2.0, 7.0, 10.0, amp=1.0, theta=33.0)
matchedFilter = gFunc1(xpos, ypos)
fig = plt.figure(figsize=(8, 8))
plt.imshow(matchedFilter, origin='lower')
plt.title('Filter tuned to source A')

In [ ]:
filteredSignal = apply_matched_filter(matchedFilter, noisySignal)
fig = plt.figure(figsize=(8, 8))
plt.imshow(filteredSignal, origin='lower')
plt.title('Matched-filter response for source A')

In [ ]:
gFunc1 = generalGauss2d(imgSize / 2.0, imgSize / 2.0, 20.0, 15.0, amp=0.3, theta=0.0)
matchedFilter = gFunc1(xpos, ypos)
fig = plt.figure(figsize=(8, 8))
plt.imshow(matchedFilter, origin='lower')
plt.title('Filter tuned to source B')

In [ ]:
filteredSignal = apply_matched_filter(matchedFilter, noisySignal)
fig = plt.figure(figsize=(8, 8))
plt.imshow(filteredSignal, origin='lower')
plt.title('Matched-filter response for source B')

In [ ]:
gFunc1 = generalGauss2d(imgSize / 2.0, imgSize / 2.0, 2.0, 3.0, amp=2.0, theta=45.0)
matchedFilter = gFunc1(xpos, ypos)
fig = plt.figure(figsize=(8, 8))
plt.imshow(matchedFilter, origin='lower')
plt.title('Filter tuned to source C')

In [ ]:
filteredSignal = apply_matched_filter(matchedFilter, noisySignal)
fig = plt.figure(figsize=(8, 8))
plt.imshow(filteredSignal, origin='lower')
plt.title('Matched-filter response for source C')

***
